# MLE-Predictive Rank Stability

## 1. Configure the Python Environment

These figures summarize repeated-experiment score variability conditional on fixed optimized parameters for 60 PDZ, TrpB, and NucB runs. The intervals do not represent parameter uncertainty or uncertainty in the optimized estimator.

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "analysis" / "figures.py").is_file():
    if (ROOT / "figures.py").is_file():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Run this notebook from the repository root or analysis directory.")
sys.path.insert(0, str(ROOT))
figures = importlib.import_module("analysis.figures")

SAMPLING = "mle_predictive"
TABLE_DIR = ROOT / "analysis" / "tables" / SAMPLING
FIGURE_DIR = ROOT / "analysis" / "figures" / SAMPLING
MAIN_K = 100
FAMILIES = ("pdz", "trpb", "nuc")
REPRESENTATIVE_IDS = (
    "pdz-cis-mle-predictive-lomax-exponential",
    "trpb-libd-mle-predictive-gamma-exponential",
    "nuc-g1-mle-predictive-exponential",
)
figures.set_paper_style()

## 2. Load or Generate Input Data

The notebook reads only the numerical tables generated by `analysis/run.py`. K50 is outside this analysis scope. Count-support relationships describe conditional predictive variation and should not be interpreted causally.

In [ ]:
tables = figures.load_all_tables(TABLE_DIR)
runs = tables["runs"]

assert len(runs) == 60
assert runs.groupby("family").size().to_dict() == {"nuc": 7, "pdz": 19, "trpb": 34}
assert runs["landscape"].nunique() == 19
assert set(runs["sampling"]) == {SAMPLING}
assert set(REPRESENTATIVE_IDS) <= set(runs["run_id"])
assert tables["mechanism_inventory"]["landscape"].nunique() == 19

display(runs)

## 3. Implement the Core Workflow

Every figure is rendered from tidy tables, saved as PDF and PNG, and then closed to bound notebook memory.

In [ ]:
generated_paths = []


def render(figure, stem):
    display(figure)
    paths = figures.save_figure(figure, FIGURE_DIR, stem)
    generated_paths.extend(paths)
    plt.close(figure)
    return paths


def stem_for(family):
    return figures.FAMILY_FILE_STEMS[family]

## 4. Visualize Results

Figures 1-3 describe external-order calibration, sampled rank intervals, and top-K decision stability across repeated count experiments conditional on each optimized fit.

In [ ]:
render(
    figures.figure_reported_order(tables, cohort_top_n=MAIN_K, example_id=REPRESENTATIVE_IDS[0]),
    "figure1_reported_order",
)
render(
    figures.figure_rank_uncertainty(
        tables,
        representative_ids=REPRESENTATIVE_IDS,
        main_k=MAIN_K,
    ),
    "figure2_rank_uncertainty",
)
render(
    figures.figure_top_k_stability(
        tables,
        representative_ids=REPRESENTATIVE_IDS,
        main_k=MAIN_K,
    ),
    "figure3_top_k_stability",
)

### Mechanism and Count-Support Views

Figure 4 links experimental support, conditional predictive score precision, and rank resolution. Figure 5 is an exploratory rank-space sensitivity view. Figures 6.1-6.3 show the same run-variant rows with complementary renderers; repeated rows across fitted models are not independent biological replicates.

In [ ]:
render(
    figures.figure_ranking_mechanisms(tables),
    "figure4_ranking_mechanisms",
)
render(
    figures.figure_rank_space_sensitivity(tables),
    "figure5_rank_space_sensitivity",
)
for renderer, stem in (
    ("scatter", "figure6_1_count_uncertainty_scatter"),
    ("datashader", "figure6_2_count_uncertainty_datashader"),
    ("hexbin", "figure6_3_count_uncertainty_hexbin"),
):
    render(figures.figure_count_uncertainty(tables, renderer=renderer), stem)

### Family-Resolved Views

Figures 7-11 retain model-specific rows to show where count support, predictive score geometry, and decision-boundary uncertainty differ across PDZ, TrpB, and NucB landscapes.

In [ ]:
for family in FAMILIES:
    stem = stem_for(family)
    render(
        figures.figure_count_score_uncertainty_by_analysis(tables, family=family),
        f"figure7_1_{stem}_count_score_uncertainty",
    )
    render(
        figures.figure_count_rank_uncertainty_by_analysis(tables, family=family),
        f"figure7_2_{stem}_count_rank_uncertainty",
    )
    render(
        figures.figure_functionality_distributions(
            runs,
            tables["functionality_scores"],
            family=family,
            top_k=MAIN_K,
        ),
        f"figure8_{stem}_functionality",
    )
    render(
        figures.figure_fitness_uncertainty_by_analysis(tables, family=family),
        f"figure9_{stem}_fitness_uncertainty",
    )
    render(
        figures.figure_summary_why_by_analysis(tables, family=family),
        f"figure10_{stem}_summary_why",
    )
    render(
        figures.figure_count_score_corner_by_analysis(tables, family=family),
        f"figure11_{stem}_count_score_corner",
    )

## 5. Validate the Implementation

Confirm that every expected figure was written in both formats and is nonempty.

In [ ]:
expected_stems = {
    "figure1_reported_order",
    "figure2_rank_uncertainty",
    "figure3_top_k_stability",
    "figure4_ranking_mechanisms",
    "figure5_rank_space_sensitivity",
    "figure6_1_count_uncertainty_scatter",
    "figure6_2_count_uncertainty_datashader",
    "figure6_3_count_uncertainty_hexbin",
}
for family in FAMILIES:
    stem = stem_for(family)
    expected_stems.update(
        {
            f"figure7_1_{stem}_count_score_uncertainty",
            f"figure7_2_{stem}_count_rank_uncertainty",
            f"figure8_{stem}_functionality",
            f"figure9_{stem}_fitness_uncertainty",
            f"figure10_{stem}_summary_why",
            f"figure11_{stem}_count_score_corner",
        }
    )

assert len(expected_stems) == 26
assert {path.stem for path in generated_paths} == expected_stems
assert len(generated_paths) == 2 * len(expected_stems)
assert {path.suffix for path in generated_paths} == {".pdf", ".png"}
assert all(path.is_file() and path.stat().st_size > 0 for path in generated_paths)